# A* search – from games to industry

In the P vs NP notebook we saw that the *shortest path* problem is in P – but "polynomial" does not mean "free":
a game with 500 units or a warehouse with 200 robots re-plans thousands of paths per second, and a country-sized road network
has $10^7$ junctions. A* is how a P problem is made fast enough in practice: Dijkstra's algorithm plus a **heuristic**.

In the first lab we saw A* on a five-node toy graph. Here we use it on problems where it really matters:

1. **Game:** pathfinding on a tile map (the same algorithm moves units in StarCraft, Age of Empires, Factorio, ...).
2. **Game:** the 8-puzzle – A* in a state space without any map.
3. **Industry:** order picking in a warehouse – A* computes distances, TSP orders the picks.
4. **Industry:** navigation on a road network – how a GPS finds the route.

The key idea of A*: always continue from the candidate with the smallest

$$f(n) = g(n) + h(n)$$

where $g(n)$ is the cost already paid from the start and $h(n)$ is a **heuristic** – an estimate of the cost still to go.
If $h$ never overestimates (it is *admissible*), A* finds the optimal path – and the better the estimate,
the fewer nodes it has to look at.

*How to read the code:* read the comments; the exact Python syntax is secondary.

In [ ]:
import heapq                       # a "priority queue": a pile from which we always take the smallest item quickly
import math                        # square roots, infinity
import itertools                   # combinations / permutations
from time import perf_counter      # stopwatch

import numpy as np                 # tables of numbers (the maps are 2-D tables)
import pandas as pd                # nice result tables
import matplotlib.pyplot as plt    # pictures
import networkx as nx              # graphs (the road network)

## A generic A* implementation

The function does not care whether a *node* is a grid cell, a puzzle configuration or a road junction –
it only needs a `neighbors(node)` function returning `(neighbor, step_cost)` pairs and a heuristic `h(node)`.

* `heuristic = lambda n: 0` turns A* into **Dijkstra's algorithm** (uniform cost search),
* `weight > 1` gives **weighted A*** – faster, but the path may be up to `weight`× longer than optimal.

In [ ]:
def a_star(start, goal, neighbors, heuristic, weight: float = 1.0):
    """
    Generic A* search.

    Parameters:
        start, goal : the two nodes (anything that can be compared and used as a dictionary key)
        neighbors   : function node -> list of (neighbor, step_cost)   ("where can I go from here and what does it cost")
        heuristic   : function node -> estimated cost to the goal      ("how far is it still, roughly")
        weight      : multiplier of the heuristic (1 = classic A*, 0 = Dijkstra, >1 = weighted A*)

    Returns:
        path (list of nodes), cost (float), expanded (set of nodes we "opened" = took out of the queue)
    """
    # The queue of candidates. Each entry is (f, g, node); heapq always gives us the entry with the smallest f.
    open_set = [(weight * heuristic(start), 0.0, start)]
    came_from = {start: None}      # for every node: from which node did we reach it (to rebuild the path at the end)
    g = {start: 0.0}               # for every node: the cheapest known cost from the start
    expanded = set()               # nodes we have already processed ("closed set")

    while open_set:                                        # while there are candidates
        _, g_current, current = heapq.heappop(open_set)    # take the most promising one (smallest f)
        if current in expanded:                            # an outdated duplicate entry -> ignore it
            continue
        expanded.add(current)                              # mark as processed

        if current == goal:                                # reached the goal -> rebuild the path backwards
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            return path[::-1], g_current, expanded         # reverse it so that it starts at `start`

        for neighbor, step_cost in neighbors(current):     # look at all neighbours of the current node
            if neighbor in expanded:                       # already processed -> nothing better can come
                continue
            g_new = g_current + step_cost                  # cost to reach the neighbour via the current node
            if g_new < g.get(neighbor, math.inf):          # cheaper than anything known so far?
                g[neighbor] = g_new                        # remember the better cost ...
                came_from[neighbor] = current              # ... and where it came from
                heapq.heappush(open_set, (g_new + weight * heuristic(neighbor), g_new, neighbor))   # add as candidate

    return None, math.inf, expanded                        # queue empty and goal not reached -> no path exists

## 1. Game: pathfinding on a tile map

A unit has to walk from the start (green) to the goal (gold). Walls are black.
The map contains a classic trap – a U-shaped wall opening towards the start.

In [ ]:
def make_game_map(rows: int = 25, cols: int = 40, seed: int = 7) -> np.ndarray:
    """A 2-D table of tiles: 0 = free tile, 1 = wall."""
    grid = np.zeros((rows, cols), dtype=int)
    rng = np.random.default_rng(seed)             # random generator with a fixed seed -> the same map every time

    # a few random rocks: about 12 % of tiles become walls
    rocks = rng.random((rows, cols)) < 0.12
    grid[rocks] = 1

    # the U-shaped trap (opening to the left, i.e. towards the start)
    grid[5:20, 22] = 1          # vertical wall (the bottom of the U)
    grid[5, 12:23] = 1          # top arm
    grid[19, 12:23] = 1         # bottom arm
    grid[6:19, 13:22] = 0       # keep the inside of the U free

    return grid


def grid_neighbors_4(grid: np.ndarray):
    """Returns a neighbour function for 4-directional movement (up/down/left/right) with cost 1 per step."""
    rows, cols = grid.shape

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):        # the four directions
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 0:   # inside the map and not a wall
                yield (nr, nc), 1
    return neighbors


def show_search(grid, start, goal, path, expanded, title, ax=None):
    """Draw the map: walls black, expanded tiles light blue, the path red, start green, goal gold."""
    ax = ax or plt.gca()
    image = np.ones(grid.shape + (3,))            # an RGB picture, white everywhere
    image[grid == 1] = (0.1, 0.1, 0.1)            # walls
    for cell in expanded:
        image[cell] = (0.7, 0.85, 1.0)            # tiles the algorithm looked at
    if path:
        for cell in path:
            image[cell] = (0.9, 0.2, 0.2)         # the found path
    image[start] = (0.1, 0.7, 0.1)
    image[goal] = (1.0, 0.8, 0.0)
    ax.imshow(image, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


game_map = make_game_map()
start, goal = (12, 3), (12, 36)                  # (row, column)
game_map[start] = game_map[goal] = 0             # make sure start and goal are not walls

plt.figure(figsize=(8, 5))
show_search(game_map, start, goal, [], set(), "The map")
plt.show()

### Which heuristic? Compare the number of expanded tiles

All heuristics below are admissible for 4-directional movement, but Manhattan distance is the tightest estimate.
"Greedy best-first" uses (almost) only the heuristic – it is fast, but the U-trap fools it into a long detour.

In [ ]:
# Distance estimates between two tiles a = (row, col) and b = (row, col):
def manhattan(a, b): return abs(a[0] - b[0]) + abs(a[1] - b[1])      # rows + columns  (exact on an empty 4-direction grid)
def euclidean(a, b): return math.hypot(a[0] - b[0], a[1] - b[1])     # straight line   (always <= Manhattan)
def chebyshev(a, b): return max(abs(a[0] - b[0]), abs(a[1] - b[1]))  # max of the two  (always <= Euclidean)
def zero(a, b):      return 0                                        # no estimate at all = Dijkstra

heuristics = {
    "Dijkstra (h = 0)":      (zero, 1),
    "A* Chebyshev":          (chebyshev, 1),
    "A* Euclidean":          (euclidean, 1),
    "A* Manhattan":          (manhattan, 1),
    "Weighted A* (w = 3)":   (manhattan, 3),        # f = g + 3h
    "Greedy best-first":     (manhattan, 1000),     # f ≈ h only
}

neighbors = grid_neighbors_4(game_map)
results, runs = [], {}
for name, (h, w) in heuristics.items():
    t0 = perf_counter()
    # lambda n, h=h: h(n, goal)  -> "the heuristic of tile n is its distance estimate to the goal"
    path, cost, expanded = a_star(start, goal, neighbors, lambda n, h=h: h(n, goal), weight=w)
    runs[name] = (path, expanded)
    results.append({"algorithm": name, "expanded tiles": len(expanded), "path length": cost,
                    "time [ms]": 1000 * (perf_counter() - t0)})

pd.DataFrame(results).set_index("algorithm")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for ax, (name, (path, expanded)) in zip(axes.flat, runs.items()):
    show_search(game_map, start, goal, path, expanded, f"{name}\nexpanded: {len(expanded)}, length: {len(path) - 1}", ax)
plt.tight_layout()
plt.show()

### Weighted A*: trading optimality for speed

In games we often do not need *the* shortest path – a path that is 10 % longer but found 5× faster is a great deal
when 500 units re-plan every frame. Weighted A* with $f = g + w \cdot h$ guarantees a path at most $w$× longer than optimal.

In [ ]:
rows = []
for w in [1, 1.2, 1.5, 2, 3, 5, 10]:
    path, cost, expanded = a_star(start, goal, neighbors, lambda n: manhattan(n, goal), weight=w)
    rows.append({"weight w": w, "expanded tiles": len(expanded), "path length": cost})
weighted = pd.DataFrame(rows).set_index("weight w")
weighted["% longer than optimal"] = 100 * (weighted["path length"] / weighted["path length"].iloc[0] - 1)
weighted

## 2. Game: the 8-puzzle

No map here – the *state* is the arrangement of tiles and a *move* slides one tile into the empty spot.
The state space has $9!/2 = 181\,440$ reachable configurations. We solve the hardest possible instance (31 moves)
with three heuristics and count the expanded states.

* `h = 0` – breadth-first search (every move costs 1, so Dijkstra = BFS),
* *misplaced tiles* – number of tiles not in their goal position,
* *Manhattan distance* – sum of the distances of each tile from its goal position.

In [ ]:
# A state is a tuple of 9 numbers read row by row; 0 = the empty spot.
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)      # 1 2 3 / 4 5 6 / 7 8 _
HARDEST = (8, 6, 7, 2, 5, 4, 3, 0, 1)   # needs 31 moves


def puzzle_neighbors(state):
    """All states reachable by one move (sliding a neighbouring tile into the empty spot). Every move costs 1."""
    i = state.index(0)                  # position of the empty spot (0..8)
    r, c = divmod(i, 3)                 # its row and column
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):       # the tile below / above / right / left of the spot
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:                     # must be inside the 3x3 board
            j = nr * 3 + nc
            s = list(state)
            s[i], s[j] = s[j], s[i]                         # swap the tile and the empty spot
            yield tuple(s), 1


def h_misplaced(state):
    """Number of tiles that are not where they should be (the empty spot is not counted)."""
    return sum(1 for i, tile in enumerate(state) if tile and tile != i + 1)


def h_manhattan(state):
    """Sum over tiles of (rows + columns) between the tile's current position and its goal position."""
    return sum(abs(i // 3 - (tile - 1) // 3) + abs(i % 3 - (tile - 1) % 3)
               for i, tile in enumerate(state) if tile)


def show_state(state):
    """Print the board as 3 lines."""
    return "\n".join(" ".join(str(t) if t else "." for t in state[r * 3:(r + 1) * 3]) for r in range(3))


print(show_state(HARDEST), "\n")

rows = []
for name, h in [("BFS (h = 0)", lambda s: 0), ("A* misplaced tiles", h_misplaced), ("A* Manhattan", h_manhattan)]:
    t0 = perf_counter()
    path, cost, expanded = a_star(HARDEST, GOAL, puzzle_neighbors, h)
    rows.append({"algorithm": name, "expanded states": len(expanded), "moves": int(cost),
                 "time [s]": perf_counter() - t0})
pd.DataFrame(rows).set_index("algorithm")

All three find the optimal 31-move solution – but the good heuristic looks at a hundred times fewer states.
The same trick (A* / IDA* with a *pattern database* heuristic) solves the 15-puzzle and the Rubik's cube optimally.

## 3. Industry: order picking in a warehouse (A* + TSP)

A picker (or an AGV robot) starts at the dock, has to collect 6 items from the shelves and return.
Around the packing area the corridors are congested, so a step there costs 3× more.

This is exactly the problem a warehouse management system solves thousands of times a day, and it combines
both algorithms of this lab:

1. **A\*** computes the walking distance between every pair of pick locations (the map has obstacles, so Euclidean distance is not enough),
2. **TSP** decides in which order to visit them.

In [ ]:
def make_warehouse(rows: int = 22, cols: int = 31):
    """The warehouse map (0 = corridor, 1 = shelf rack) and the cost of stepping onto each tile."""
    grid = np.zeros((rows, cols), dtype=int)
    for c in (3, 4, 8, 9, 13, 14, 18, 19, 23, 24):           # racks are vertical blocks two columns wide
        grid[3:19, c] = 1
    cost = np.ones((rows, cols))                             # normally a step costs 1 ...
    cost[rows - 3:, :11] = 3                                 # ... but 3 in the congested packing area (bottom left)
    return grid, cost


def warehouse_neighbors(grid, cost):
    """Neighbour function: 4 directions, the cost of a step is the cost of the tile we step onto."""
    rows, cols = grid.shape

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 0:
                yield (nr, nc), cost[nr, nc]
    return neighbors


warehouse, step_cost = make_warehouse()
dock = (20, 1)
picks = [(5, 5), (16, 7), (9, 12), (14, 15), (6, 20), (17, 25)]     # (row, column) of the six items
locations = [dock] + picks
labels = ["dock"] + [f"P{i + 1}" for i in range(len(picks))]


def show_warehouse(path=None, title="Warehouse"):
    image = np.ones(warehouse.shape + (3,))
    image[step_cost == 3] = (1.0, 0.9, 0.8)       # congested area: orange
    image[warehouse == 1] = (0.35, 0.25, 0.15)    # racks: brown
    if path:
        for cell in path:
            image[cell] = (0.9, 0.2, 0.2)         # route: red
    plt.figure(figsize=(9, 6))
    plt.imshow(image, interpolation="nearest")
    for label, (r, c) in zip(labels, locations):
        plt.plot(c, r, "o", color="gold" if label == "dock" else "royalblue", markersize=14)
        plt.text(c, r, label, ha="center", va="center", fontsize=7, fontweight="bold")
    plt.title(title); plt.xticks([]); plt.yticks([])
    plt.show()


show_warehouse(title="Warehouse: racks (brown), congested packing area (orange), pick locations (blue)")

In [ ]:
# Step 1: pairwise distances with A*  (Manhattan is admissible because every step costs at least 1)
neighbors = warehouse_neighbors(warehouse, step_cost)
k = len(locations)
distance_matrix = np.zeros((k, k))
paths = {}
for i, j in itertools.combinations(range(k), 2):          # every pair of locations (21 pairs)
    path, cost, _ = a_star(locations[i], locations[j], neighbors, lambda n, g=locations[j]: manhattan(n, g))
    distance_matrix[i, j] = distance_matrix[j, i] = cost  # the walking distance goes into the matrix (both directions)
    paths[i, j] = path                                    # keep the actual path to draw the route later
    paths[j, i] = path[::-1]

pd.DataFrame(distance_matrix, index=labels, columns=labels)

In [ ]:
# Step 2: order of the picks = TSP on the distance matrix (6 picks -> 720 permutations, brute force is fine)
def tour_length(order):
    """Length of the tour dock -> order of picks -> dock."""
    tour = [0] + list(order) + [0]
    return sum(distance_matrix[a, b] for a, b in zip(tour, tour[1:]))

best_order = min(itertools.permutations(range(1, k)), key=tour_length)     # try all 720 orders, keep the shortest

# greedy nearest-neighbour order for comparison
remaining, current, greedy_order = set(range(1, k)), 0, []
while remaining:
    current = min(remaining, key=lambda j: distance_matrix[current, j])     # nearest unvisited pick
    remaining.remove(current)
    greedy_order.append(current)

print("Optimal order:", " -> ".join(labels[i] for i in [0, *best_order, 0]), f"  length {tour_length(best_order):.0f}")
print("Greedy order: ", " -> ".join(labels[i] for i in [0, *greedy_order, 0]), f"  length {tour_length(greedy_order):.0f}")

# Step 3: stitch the A* paths together and draw the route
tour = [0] + list(best_order) + [0]
full_route = [cell for a, b in zip(tour, tour[1:]) for cell in paths[a, b]]
show_warehouse(full_route, title=f"Optimal picking route, length {tour_length(best_order):.0f}")

## 4. Industry: navigation on a road network

A road map is a graph with coordinates: crossings are nodes, roads are edges weighted by their length.
Dijkstra's algorithm explores in all directions like a growing circle; A* with the straight-line (Euclidean)
distance as heuristic explores mostly *towards* the destination.

(Real navigation systems add more tricks on top – contraction hierarchies, landmarks – but the core is A*.)

In [ ]:
# A random "road network": 600 junctions, roads between junctions closer than 0.075
roads = nx.random_geometric_graph(600, 0.075, seed=4)
roads = roads.subgraph(max(nx.connected_components(roads), key=len)).copy()   # keep only the connected part
coordinates = nx.get_node_attributes(roads, "pos")
for u, v in roads.edges():
    roads[u][v]["weight"] = math.dist(coordinates[u], coordinates[v])         # road length = straight-line distance

origin = min(roads, key=lambda n: math.dist(coordinates[n], (0.05, 0.05)))        # junction near the bottom-left corner
destination = min(roads, key=lambda n: math.dist(coordinates[n], (0.95, 0.95)))   # junction near the top-right corner


def road_neighbors(u):
    """Neighbouring junctions and the length of the road to them."""
    return ((v, roads[u][v]["weight"]) for v in roads[u])


def straight_line(u):
    """Heuristic: straight-line distance to the destination (never longer than the real road distance)."""
    return math.dist(coordinates[u], coordinates[destination])


fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, (name, h) in zip(axes, [("Dijkstra", lambda n: 0), ("A* (straight-line heuristic)", straight_line)]):
    t0 = perf_counter()
    path, cost, expanded = a_star(origin, destination, road_neighbors, h)
    elapsed = perf_counter() - t0
    nx.draw_networkx_edges(roads, coordinates, ax=ax, edge_color="lightgray", width=0.5)
    nx.draw_networkx_nodes(roads, coordinates, ax=ax, node_size=6, node_color="lightgray")
    nx.draw_networkx_nodes(roads, coordinates, nodelist=list(expanded), ax=ax, node_size=10, node_color="skyblue")
    nx.draw_networkx_edges(roads, coordinates, edgelist=list(zip(path, path[1:])), ax=ax, edge_color="red", width=2.5)
    nx.draw_networkx_nodes(roads, coordinates, nodelist=[origin, destination], ax=ax, node_size=80, node_color=["green", "gold"])
    ax.set_title(f"{name}\nexpanded {len(expanded)} of {roads.number_of_nodes()} junctions, route length {cost:.3f}, {1000 * elapsed:.1f} ms")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

# sanity check against the networkx implementation
print("networkx A* route length:", nx.astar_path_length(roads, origin, destination,
                                                       heuristic=lambda u, v: math.dist(coordinates[u], coordinates[v]),
                                                       weight="weight"))

## Where else is A* used?

| Domain | What is a *node*? | What is the heuristic? |
|---|---|---|
| Video games (RTS, RPG, roguelikes) | tile / navmesh polygon | straight-line or octile distance |
| Mobile robots, Mars rovers, autonomous cars | robot pose (x, y, heading) | distance to goal ignoring obstacles |
| Warehouse AGVs, order picking | grid cell | Manhattan distance |
| GPS navigation | road junction | straight-line distance / travel time at max. speed |
| Printed circuit board / chip routing | grid cell in a routing layer | Manhattan distance |
| Automated planning (logistics, spacecraft) | world state | relaxed problem (ignore some constraints) |
| Puzzles (15-puzzle, Rubik's cube) | configuration | pattern databases |
| Speech recognition, machine translation | partial sentence | estimated remaining score |

The recipe is always the same: define states, moves, costs – and find an admissible heuristic by *relaxing* the problem
(remove walls → straight-line distance, let tiles fly → Manhattan distance).

## Exercise: diagonal movement

Allow the game unit to move diagonally (8 directions) with cost $\sqrt{2}$ for a diagonal step.

1. Implement `grid_neighbors_8`.
2. Implement the *octile* heuristic $h = (\Delta_{max} - \Delta_{min}) + \sqrt{2}\,\Delta_{min}$.
3. Is the Manhattan heuristic still admissible with diagonal moves? Verify experimentally: compare the path cost
   found with Manhattan against Dijkstra (`h = 0`).

In [ ]:
def grid_neighbors_8(grid: np.ndarray):
    rows, cols = grid.shape

    def neighbors(cell):
        r, c = cell
        # YOUR CODE HERE: yield ((nr, nc), cost) for all 8 directions; straight step costs 1, diagonal step costs sqrt(2)
        yield from ()
    return neighbors


def octile(a, b):
    # YOUR CODE HERE
    return 0


# neighbors_8 = grid_neighbors_8(game_map)
# for name, h in [("Dijkstra", zero), ("Manhattan", manhattan), ("Euclidean", euclidean), ("Octile", octile)]:
#     path, cost, expanded = a_star(start, goal, neighbors_8, lambda n, h=h: h(n, goal))
#     print(f"{name:10s} expanded: {len(expanded):5d}   path cost: {cost:.3f}")